# PDF Text Extraction and Chunking

### PDF to text

PDFs contain rich formatting that needs to be extracted as clean text for AI processing. 

Modern libraries like `docling` can preserve document structure while converting to text format, making the text easier to process while maintaining semantic relationships.

Let's try a popular library, `docling`, to convert PDFs into text files:

**Run the cells below to install `docling`, process the PDF documents using it, and parse them as markdown files.**

In [ ]:
# Install docling...

## TASK 1
from pathlib import Path
from docling.document_converter import DocumentConverter

def parse_pdf(input_doc_path: Path, output_dir: Path):

    md_filepath = output_dir / f"{input_doc_path.stem}-parsed-text.md"
    
    if md_filepath.exists():
        return md_filepath
    
    doc_converter = DocumentConverter()
    conv_res = doc_converter.convert(input_doc_path)

    with md_filepath.open("w", encoding="utf-8") as md_file:
        md_file.write(conv_res.document.export_to_markdown())
    
    return md_filepath

data_folder = Path("data/pdfs")
output_dir = Path("data/parsed")
output_dir.mkdir(parents=True, exist_ok=True)

input_pdf_name = "howto-free-threading-python.pdf"
input_doc_path = data_folder / input_pdf_name
parse_pdf(input_doc_path, output_dir)

md_filepath = Path("data/parsed/howto-free-threading-python-parsed-text.md")
md_txt = md_filepath.read_text()
print(md_txt[:2000])

## TASK 2
def get_chunks_by_length_with_overlap(src_text: str, chunk_length: int = 500, overlap: int = 100) -> list[str]:
    """
    Split text into chunks of approximately `chunk_length` characters.
    """
    chunks = []
    for i in range(0, len(src_text), chunk_length):
        chunks.append(src_text[i:i + chunk_length + overlap])
    return chunks
  
from IPython.display import display, Markdown

chunks = get_chunks_by_length_with_overlap(md_txt)
display(Markdown(chunks[0]))
print(len(chunks[0]))

## TASK 3
def get_chunks_using_markers(src_text: str) -> list[str]:
    """
    Split the source text into chunks using markers.
    """
    marker = "\n##"

    # Split by marker and reconstruct with markers (except first chunk)
    parts = src_text.split(marker)
    chunks = []

    # Add first chunk if it exists and isn't empty
    if parts[0].strip():
        chunks.append(parts[0].strip())

    # Add remaining chunks with markers reattached
    for part in parts[1:]:
        if part.strip():
            chunks.append(marker + part.strip())

    return chunks
  
md_file_1 = Path("data/parsed/howto-free-threading-python-parsed-text.md")
md_text_1 = md_file_1.read_text(encoding="utf-8")
chunks = get_chunks_using_markers(md_text_1)

for chunk in chunks[:5]:
    print("\n\nChunk:" + "=" * 10 + f"\n{chunk}")

md_file_2 = Path("data/parsed/hai_ai-index-report-2025_chapter2_excerpts-parsed-text.md")
md_text_2 = md_file_2.read_text(encoding="utf-8")
chunks = get_chunks_using_markers(md_text_2)

for chunk in chunks[:5]:
    print("\n\nChunk:" + "=" * 10 + f"\n{chunk}")


In [1]:
!pip uninstall transformers -qy

ERROR: Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.10/shutil.py", line 816, in move
    os.rename(src, real_dst)
PermissionError: [Errno 13] Permission denied: '/usr/local/bin/transformers-cli' -> '/tmp/pip-uninstall-3fehvqh2/transformers-cli'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/uninstall.py", line 106, in run
    uninstall_pathset = req.uninstall(
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/req/req_install.py", line 723, in uninstall
    uninstalled_pathset.remove(auto_confirm, verbose)
  File "/usr/local/lib/python3.10/dist-packa

In [1]:
# Install docling if not already installed
!pip install -q transformers==4.56.1 numpy==1.24.4 docling==2.52.0

In [2]:
from pathlib import Path
from docling.document_converter import DocumentConverter
import os
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

def parse_pdf(input_doc_path: Path, output_dir: Path):

    md_filepath = output_dir / f"{input_doc_path.stem}-parsed-text.md"
    
    if md_filepath.exists():
        return md_filepath
    
    doc_converter = DocumentConverter()
    conv_res = doc_converter.convert(input_doc_path)

    with md_filepath.open("w", encoding="utf-8") as md_file:
        md_file.write(conv_res.document.export_to_markdown())
    
    return md_filepath

This code creates the necessary directories and calls the `parse_pdf()` function to convert the `"howto-free-threading-python.pdf"` PDF file into a markdown file.

In [3]:
data_folder = Path("data/pdfs")
output_dir = Path("data/parsed")
output_dir.mkdir(parents=True, exist_ok=True)

In [4]:
input_pdf_name = "howto-free-threading-python.pdf"
input_doc_path = data_folder / input_pdf_name
parse_pdf(input_doc_path, output_dir)

We can inspect the converted file to see what it looks like:

In [5]:
md_filepath = Path("data/parsed/howto-free-threading-python-parsed-text.md")
md_txt = md_filepath.read_text()
print(md_txt[:2000])

## Chunking

Raw text from PDFs is often too long for AI models to process effectively. Chunking breaks documents into smaller, manageable pieces while preserving context.

![images/chunking_why.png](images/chunking_why.png)

#### Chunk by text length

Fixed-length chunks are simple but can break sentences or paragraphs mid-thought. This approach is fast and predictable, making it suitable for initial processing or when document structure is uniform.

Different chunking strategies serve different use cases. 

![images/chunking_methods.png](images/chunking_methods.png)

### Chunk by text length with overlap

Overlapping chunks help maintain context across boundaries. 

When a concept spans multiple chunks, the overlap helps to capture it. This is especially important for maintaining semantic coherence in search and retrieval systems.

**Define a `get_chunks_by_length_with_overlap()` function to chunk `md_txt` using a `500` character chunk length and `100` character overlap (the defaults).**

In [6]:
def get_chunks_by_length_with_overlap(src_text: str, chunk_length: int = 500, overlap: int = 100) -> list[str]:
    """
    Split text into chunks of approximately `chunk_length` characters.
    """
    chunks = []
    for i in range(0, len(src_text), chunk_length):
        chunks.append(src_text[i:i + chunk_length + overlap])
    return chunks

In [7]:
from IPython.display import display, Markdown

chunks = get_chunks_by_length_with_overlap(____)
display(Markdown(chunks[0]))
print(len(chunks[0]))

### Chunk using markers

Using document markers (like headers) creates chunks that respect natural document boundaries. 

This approach preserves semantic structure and is ideal for documents with clear hierarchical organization like reports, manuals, or academic papers.

**Define a `get_chunks_using_markers()` function to chunk `md_text_1` by splitting on non-title headings (`"\n##"`).**

In [8]:
def get_chunks_using_markers(src_text: str) -> list[str]:
    """
    Split the source text into chunks using markers.
    """
    marker = "____"

    # Split by marker and reconstruct with markers (except first chunk)
    parts = src_text.split(marker)
    chunks = []

    # Add first chunk if it exists and isn't empty
    if parts[0].strip():
        chunks.append(parts[0].strip())

    # Add remaining chunks with markers reattached
    for part in parts[1:]:
        if part.strip():
            chunks.append(marker + part.strip())

    return chunks

In [9]:
md_file_1 = Path("data/parsed/howto-free-threading-python-parsed-text.md")
md_text_1 = md_file_1.read_text(encoding="utf-8")
chunks = get_chunks_using_markers(____)

for chunk in chunks[:5]:
    print("\n\nChunk: " + "=" * 10 + f"\n{chunk}")

### Choosing the right strategy

The best chunking strategy depends on your use case. 

Marker-based chunking excels with structured documents as you saw. But in some cases, it may not work as well.

**Apply the `get_chunks_using_markers()` function to `md_text_2` and compare the results to `md_text_1`.**

In [10]:
md_file_2 = Path("data/parsed/hai_ai-index-report-2025_chapter2_excerpts-parsed-text.md")
md_text_2 = md_file_2.read_text(encoding="utf-8")
chunks = get_chunks_using_markers(____)

for chunk in chunks[:5]:
    print("\n\nChunk:" + "=" * 10 + f"\n{chunk}")

Here, the page headers are mistakenly interpreted as headings, which confuses our structure. 

In general, the best chunking strategy for you will depend on your specific set of circumstances. But a fixed-length chunking strategy with overlap is a good default choice.